In [1]:
import os
os.listdir('.')

['run_log.txt',
 'memory',
 '.config',
 'generate_ldh_zeros.py',
 '.prompts',
 '.kernel_llm_logs_1.txt']

In [2]:
import subprocess
# Check the run_log size
result = subprocess.run(['wc', '-l', 'run_log.txt'], capture_output=True, text=True)
print(result.stdout)
# Check if cypari2 available
import importlib
for pkg in ['cypari2', 'mpmath', 'numpy']:
 try:
 m = importlib.import_module(pkg)
 print(pkg, 'OK', getattr(m, '__version__', '?'))
 except Exception as e:
 print(pkg, 'MISSING', e)

58 run_log.txt

cypari2 MISSING No module named 'cypari2'
mpmath OK 1.3.0
numpy OK 2.4.3


In [3]:
%%bash
pip install cypari2 2>&1 | tail -5

_x86_64.whl.metadata (12 kB)

17_x86_64.manylinux_2_28_x86_64.whl (9.5 MB)

x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (268 kB)

nals, cypari2


In [4]:
import cypari2
print('cypari2 OK')
# check if WORK_DIR exists; the script hardcodes a different working dir
import os
HARDCODED = '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d'
print('hardcoded exists:', os.path.exists(HARDCODED))
print('cwd:', os.getcwd())

cypari2 OK
hardcoded exists: False
cwd: /workspace/58ebda78-3b28-4304-bd4e-c15d1dcfb57a


In [5]:
# Patch the script to use the current workspace directory rather than the hardcoded one
import shutil
shutil.copy('generate_ldh_zeros.py', 'generate_ldh_zeros_local.py')

# Edit the WORK_DIR
with open('generate_ldh_zeros_local.py', 'r') as f:
 src = f.read()
src_new = src.replace(
 "WORK_DIR = '/workspace/c7eacfb8-052d-4c79-89db-dce9cc4b7e1d'",
 "WORK_DIR = '/workspace/58ebda78-3b28-4304-bd4e-c15d1dcfb57a'"
)
# Also rename log so we don't clobber existing run_log
src_new = src_new.replace("LOG_PATH = os.path.join(WORK_DIR, 'run_log.txt')",
 "LOG_PATH = os.path.join(WORK_DIR, 'run_log_local.txt')")
with open('generate_ldh_zeros_local.py', 'w') as f:
 f.write(src_new)
print('patched')

# Confirm there is no preexisting checkpoint (script will start from scratch t=1.0)
print('checkpoints_ldh exists:', os.path.exists('checkpoints_ldh'))

patched
checkpoints_ldh exists: False


In [6]:

# Launch the script as a background subprocess so we can monitor progress
import subprocess, time, os

log_path = 'run_log_local.txt'
# Ensure fresh log
if os.path.exists(log_path):
 os.remove(log_path)

proc = subprocess.Popen(
 ['python', '-u', 'generate_ldh_zeros_local.py'],
 stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
 cwd=os.getcwd(),
)
print('PID:', proc.pid)
start_wall = time.time()

PID: 71


In [7]:
import time
# wait 60 seconds then check log
time.sleep(60)
print('return code:', proc.poll())
if os.path.exists(log_path):
 with open(log_path) as f:
 print(f.read())

return code: None
[08:27:03] Starting L_DH zero generation
[08:27:03] Building lfuninit at dps=80, T_max=5200 ...
[08:27:10] lfuninit done in 6.9s
[08:27:47] Checkpoint 00250: 250 zeros, t=356.30, elapsed=0.6min, rate=6.78 z/s, n_evals=6554



In [8]:
import time
# Poll periodically until completion or until ~95 minutes elapsed
deadline = start_wall + 95 * 60
last_size = 0
while True:
 rc = proc.poll()
 now = time.time()
 if rc is not None:
 print(f'Process exited rc={rc} at elapsed {(now-start_wall)/60:.1f} min')
 break
 if now > deadline:
 print('Deadline reached, terminating')
 proc.terminate()
 break
 # Print last log line
 if os.path.exists(log_path):
 sz = os.path.getsize(log_path)
 if sz > last_size:
 with open(log_path) as f:
 lines = f.readlines()
 print(f"[poll {(now-start_wall)/60:.1f}min] tail:", lines[-1].rstrip())
 last_size = sz
 time.sleep(120)

TimeoutError: Code execution timed out after 900.0 seconds

In [9]:

# Check process status without blocking
import subprocess, os, time
# Look up the PID we saved
try:
 print('proc poll:', proc.poll())
except NameError:
 proc = None
# Find latest progress
if os.path.exists(log_path):
 with open(log_path) as f:
 lines = f.readlines()
 print('lines:', len(lines))
 print(''.join(lines[-8:]))

proc poll: -2
lines: 26
[08:37:11] PROGRESS: 1610 zeros, t=1756.70, elapsed=10.0min
[08:38:11] PROGRESS: 1674 zeros, t=1821.20, elapsed=11.0min
[08:39:11] PROGRESS: 1735 zeros, t=1879.00, elapsed=12.0min
[08:39:26] Checkpoint 01750: 1750 zeros, t=1893.10, elapsed=12.3min, rate=2.38 z/s, n_evals=39922
[08:40:12] PROGRESS: 1801 zeros, t=1941.90, elapsed=13.0min
[08:41:12] PROGRESS: 1860 zeros, t=2001.10, elapsed=14.0min
[08:42:12] PROGRESS: 1913 zeros, t=2051.10, elapsed=15.0min
[08:43:12] PROGRESS: 1964 zeros, t=2098.10, elapsed=16.0min



In [10]:
# The poll() returned -2 (SIGINT) — the cell timeout killed our notebook poll loop, which
# unfortunately terminated the child process via SIGINT propagation. Verify.
import os, signal, subprocess
result = subprocess.run(['ps', '-ef'], capture_output=True, text=True)
print(result.stdout[:3000])

PermissionError: [Errno 13] Permission denied: 'ps'

In [11]:
# Inspect /proc directly
import os
pids = [p for p in os.listdir('/proc') if p.isdigit()]
my_pid = os.getpid()
print('my pid:', my_pid)
# Look for python processes other than this one
for p in pids:
 try:
 with open(f'/proc/{p}/cmdline','rb') as f:
 cmd = f.read().replace(b'\x00', b' ').decode(errors='ignore').strip()
 if 'generate_ldh_zeros' in cmd or 'ldh' in cmd:
 print(p, cmd)
 except Exception:
 pass

my pid: 20


In [12]:
# Process was killed. But we have a checkpoint at 1750. Let's check.
import os
print(sorted(os.listdir('checkpoints_ldh')) if os.path.exists('checkpoints_ldh') else 'no ckpt dir')

['ldh_ckpt_00250.npy', 'ldh_ckpt_00250_meta.json', 'ldh_ckpt_00500.npy', 'ldh_ckpt_00500_meta.json', 'ldh_ckpt_00750.npy', 'ldh_ckpt_00750_meta.json', 'ldh_ckpt_01000.npy', 'ldh_ckpt_01000_meta.json', 'ldh_ckpt_01250.npy', 'ldh_ckpt_01250_meta.json', 'ldh_ckpt_01500.npy', 'ldh_ckpt_01500_meta.json', 'ldh_ckpt_01750.npy', 'ldh_ckpt_01750_meta.json']


In [13]:
# Good - we have checkpoint 1750. Now restart the script — it will resume from the latest checkpoint.
# Use nohup-style detachment via os.setpgrp so the child won't get a SIGINT from our cell timeouts.
import subprocess, os, time

# Make sure we use a NEW polling approach: don't use sleep loops longer than 800s in a cell.
proc2 = subprocess.Popen(
 ['python', '-u', 'generate_ldh_zeros_local.py'],
 stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
 cwd=os.getcwd(),
 start_new_session=True, # new session, detached from this kernel
)
print('PID:', proc2.pid)
start_wall2 = time.time()
time.sleep(15)
print('poll:', proc2.poll())
with open(log_path) as f:
 print(''.join(f.readlines()[-5:]))

PID: 81


poll: None
[08:43:12] PROGRESS: 1964 zeros, t=2098.10, elapsed=16.0min
[08:43:47] Starting L_DH zero generation
[08:43:47] Building lfuninit at dps=80, T_max=5200 ...
[08:43:54] lfuninit done in 6.9s
[08:43:54] Resumed from ldh_ckpt_01750.npy: 1750 zeros, t_last=1893.1000



In [14]:
# Good — resumed from checkpoint 1750. Now poll in short bursts (under 800s per cell) and don't kill child.
import time, os
def tail_log(n=5):
 with open(log_path) as f:
 lines = f.readlines()
 return ''.join(lines[-n:])

# 1st burst: wait ~600s
time.sleep(600)
print('poll:', proc2.poll())
print('elapsed:', (time.time()-start_wall2)/60, 'min')
print(tail_log(8))

poll: None
elapsed: 10.347843412558237 min
[08:48:23] Checkpoint 02000: 2000 zeros, t=2129.80, elapsed=4.5min, rate=7.43 z/s, n_evals=5368
[08:48:55] PROGRESS: 2026 zeros, t=2153.40, elapsed=5.0min
[08:49:55] PROGRESS: 2072 zeros, t=2196.80, elapsed=6.0min
[08:50:55] PROGRESS: 2116 zeros, t=2237.90, elapsed=7.0min
[08:51:56] PROGRESS: 2166 zeros, t=2284.50, elapsed=8.0min
[08:52:56] PROGRESS: 2220 zeros, t=2331.70, elapsed=9.0min
[08:53:30] Checkpoint 02250: 2250 zeros, t=2359.70, elapsed=9.6min, rate=3.91 z/s, n_evals=10667
[08:53:56] PROGRESS: 2271 zeros, t=2380.40, elapsed=10.0min



In [15]:
import time
# 2nd burst: ~700s
time.sleep(700)
print('poll:', proc2.poll())
print('elapsed:', (time.time()-start_wall2)/60, 'min')
print(tail_log(10))

poll: None
elapsed: 22.10596469640732 min
[08:56:57] PROGRESS: 2411 zeros, t=2507.30, elapsed=13.0min
[08:57:57] PROGRESS: 2431 zeros, t=2523.20, elapsed=14.0min
[08:58:57] PROGRESS: 2450 zeros, t=2540.70, elapsed=15.0min
[08:59:59] PROGRESS: 2471 zeros, t=2557.90, elapsed=16.1min
[09:01:00] PROGRESS: 2490 zeros, t=2576.90, elapsed=17.1min
[09:01:30] Checkpoint 02500: 2500 zeros, t=2585.00, elapsed=17.6min, rate=2.37 z/s, n_evals=15920
[09:02:02] PROGRESS: 2510 zeros, t=2595.20, elapsed=18.1min
[09:03:02] PROGRESS: 2529 zeros, t=2612.30, elapsed=19.1min
[09:04:03] PROGRESS: 2549 zeros, t=2630.40, elapsed=20.1min
[09:05:05] PROGRESS: 2569 zeros, t=2648.20, elapsed=21.2min



In [16]:
import time
# 3rd burst: ~700s
time.sleep(700)
print('poll:', proc2.poll())
print('elapsed:', (time.time()-start_wall2)/60, 'min')
print(tail_log(10))

TimeoutError: Code execution timed out after 616.0 seconds